In [1]:
! uv pip install langchain-community

Using Python 3.13.11 environment at: .env_rag
Audited 1 package in 87ms


In [2]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    path="./text_files",
    glob="*.txt",
    loader_cls=TextLoader
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")

Loaded 3 documents


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 185 chunks


In [4]:
! uv pip install langchain-chroma

Using Python 3.13.11 environment at: .env_rag
Audited 1 package in 17ms


In [5]:
! uv pip install -U langchain-huggingface

Using Python 3.13.11 environment at: .env_rag
Resolved 33 packages in 206ms                                        
Prepared 1 package in 4ms                                                
Uninstalled 1 package in 10ms
Installed 1 package in 24ms6.2                              
 - huggingface-hub==1.5.0
 + huggingface-hub==0.36.2


In [6]:
! uv pip install sentence-transformers transformers accelerate

Using Python 3.13.11 environment at: .env_rag
Resolved 60 packages in 154ms                                        
Uninstalled 1 package in 3ms
Installed 1 package in 22ms.0                               
 - huggingface-hub==0.36.2
 + huggingface-hub==1.5.0


In [7]:
# Initialize a ChromaDB Connection
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Initialize the database connection
# If database exist, it will connect with the collection_name and persist_directory
# Otherwise a new collection will be created
db = Chroma(collection_name="vector_database", 
            embedding_function=embeddings, 
            persist_directory="./chroma_db")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
# Insert chunks in the vector database

db.add_documents(documents=chunks)

['9caa7343-c78e-4f70-a215-50644af8e69e',
 '38818e77-c578-43e8-9f0a-5aa505fc20f4',
 '62d65dbc-6996-4e61-b878-611cd1c2dba7',
 'f55c59d1-6a74-4530-9d78-a9dca2233daf',
 '8d8c08bc-2cfa-437c-a1fa-8d1d280e6051',
 '03acd6bf-48d5-4711-958c-3a16aeedf8e2',
 '9afc56b8-0e8d-411d-82af-e0a23c08bb37',
 '7afc9070-3879-4dff-85cb-37e495c57f29',
 '20a96f30-6e4b-415b-aeb3-8bff390c0df7',
 '7f7f2417-1209-4c67-a97c-2f4efe41ba7e',
 'df236793-babd-4620-aacf-1b69be71d9e0',
 'ab7c3543-c9c5-4633-abf9-aa36b7f28175',
 'eaa0b6ee-dc64-4719-b8b7-f7dc8e33a4c8',
 '65f99812-d2c3-4fcd-ae71-398fc165001b',
 'b4f6f5db-d109-4f8f-847b-0f72d055b6e5',
 'adc02d24-2afe-4a98-a07f-ba6a5964acd3',
 '889286f5-28e0-4ee8-a02c-c0660f55eba2',
 'ed6ed12e-c76f-4a3a-a62b-0a18885e6038',
 'd0150cc4-7111-4e07-98bd-1a093b6344b8',
 '67caee22-dd77-40e0-b437-18f9fd4995ff',
 '6ec10ebb-1aa8-474f-b507-5f5dc07313fa',
 'b00f8a52-13e1-406c-956a-4ced8eb6df68',
 '5bd62446-8626-41bc-96d0-4961d39fe804',
 '9b3ecde1-394c-432e-a90f-9ee0b900faff',
 '7bcceaa6-c5ed-

In [9]:
print(len(db.get()["ids"]))

740


In [10]:
# Step 1: Initialize the Chroma DB Connection

from langchain_chroma import Chroma

db = Chroma(collection_name="vector_database", 
            embedding_function=embeddings, 
            persist_directory="./chroma_db")

print(len(db.get()["ids"]))

740


In [11]:
# Step 2: Create a Retriever Object 

retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [12]:
# Step 3: Initialize a Chat Prompt Template

from langchain_core.prompts import ChatPromptTemplate

PROMPT_TEMPLATE = """
Answer the question based only on the following context:
{context}
Answer the question based on the above context: {question}.
Provide a detailed answer.
Don’t justify your answers.
Don’t give information not mentioned in the CONTEXT INFORMATION.
Do not say "according to the context" or "mentioned in the context" or similar.
"""

prompt_template = ChatPromptTemplate(
    messages=[
        PROMPT_TEMPLATE
    ]
)

prompt_template

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question based only on the following context:\n{context}\nAnswer the question based on the above context: {question}.\nProvide a detailed answer.\nDon’t justify your answers.\nDon’t give information not mentioned in the CONTEXT INFORMATION.\nDo not say "according to the context" or "mentioned in the context" or similar.\n'), additional_kwargs={})])

In [21]:
# Step 4: Initialize a Chat Model

from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-base"
)
chat_model = HuggingFacePipeline(pipeline=hf_pipeline)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'ExaoneMoeForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalL

In [22]:
# Step 5: Initialize an Output Parser

from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [23]:
generator_chain = prompt_template | chat_model | parser

In [24]:
# Helper function to join the retrieved chunks

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [25]:
# Step 6: Define a RAG Chain

from langchain_core.runnables import RunnablePassthrough

rag_chain = {
    "context": retriever | format_docs, 
    "question": RunnablePassthrough()
} | generator_chain

In [26]:
# Invoke the Chain

query = 'What is Alzheimer’s disease?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nAnswer the question based only on the following context:\nAlzheimer\'s disease (AD) is a neurodegenerative disease and is the most common form of dementia, accounting for around 60–70% of cases. \nThe most common early symptom is difficulty in remembering recent events. \nAs the disease advances, symptoms can include problems with language, disorientation (including easily getting lost), mood swings, loss of motivation, self-neglect, and behavioral issues. \nAs a person\'s condition declines, they often withdraw from family and society.\n\nAlzheimer\'s disease (AD) is a neurodegenerative disease and is the most common form of dementia, accounting for around 60–70% of cases. \nThe most common early symptom is difficulty in remembering recent events. \nAs the disease advances, symptoms can include problems with language, disorientation (including easily getting lost), mood swings, loss of motivation, self-neglect, and behavioral issues. \nAs a person\'s condition declines, they

In [27]:
query = 'What is typically the earliest symptom of Alzheimer’s disease?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nAnswer the question based only on the following context:\nThe first symptoms are often mistakenly attributed to aging or stress. Detailed neuropsychological testing can reveal mild cognitive difficulties up to eight years before a person fulfills the clinical criteria for diagnosis of Alzheimer\'s disease. These early symptoms can affect the most complex activities of daily living. The most noticeable deficit is short term memory loss, which shows up as difficulty in remembering recently learned facts and inability to acquire new information.\n\nThe first symptoms are often mistakenly attributed to aging or stress. Detailed neuropsychological testing can reveal mild cognitive difficulties up to eight years before a person fulfills the clinical criteria for diagnosis of Alzheimer\'s disease. These early symptoms can affect the most complex activities of daily living. The most noticeable deficit is short term memory loss, which shows up as difficulty in remembering recently lea

In [21]:
query = 'What are neurofibrillary tangles?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nAnswer the question based only on the following context:\nAβ plaques are dense, mostly insoluble deposits of amyloid beta peptide and cellular material outside and around neurons. Neurofibrillary tangles are aggregates of the microtubule-associated protein tau which has become hyperphosphorylated and accumulates inside neurons. Although many older individuals develop some plaques and tangles as a consequence of aging, the brains of people with Alzheimer\'s disease have a greater number of them in specific brain regions.\n\nAβ plaques are dense, mostly insoluble deposits of amyloid beta peptide and cellular material outside and around neurons. Neurofibrillary tangles are aggregates of the microtubule-associated protein tau which has become hyperphosphorylated and accumulates inside neurons. Although many older individuals develop some plaques and tangles as a consequence of aging, the brains of people with Alzheimer\'s disease have a greater number of them in specific brain re

In [22]:
query = 'What is the role of the APOE gene?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nAnswer the question based only on the following context:\nThe strongest genetic risk factor for sporadic Alzheimer\'s disease is APOEε4. APOEε4 is one of four alleles of apolipoprotein E (APOE). APOE plays a major role in lipid-binding proteins in lipoprotein particles and the ε4 allele disrupts this function. Between 40% and 80% of people with Alzheimer\'s disease possess at least one APOEε4 allele. The APOEε4 allele increases the risk of the disease by three times in heterozygotes and by 15 times in homozygotes. Like many human diseases, environmental\n\nThe strongest genetic risk factor for sporadic Alzheimer\'s disease is APOEε4. APOEε4 is one of four alleles of apolipoprotein E (APOE). APOE plays a major role in lipid-binding proteins in lipoprotein particles and the ε4 allele disrupts this function. Between 40% and 80% of people with Alzheimer\'s disease possess at least one APOEε4 allele. The APOEε4 allele increases the risk of the disease by three times in heterozygot

In [23]:
query = 'How is Alzheimer’s disease diagnosed?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nAnswer the question based only on the following context:\nAlzheimer\'s disease (AD) is a neurodegenerative disease and is the most common form of dementia, accounting for around 60–70% of cases. \nThe most common early symptom is difficulty in remembering recent events. \nAs the disease advances, symptoms can include problems with language, disorientation (including easily getting lost), mood swings, loss of motivation, self-neglect, and behavioral issues. \nAs a person\'s condition declines, they often withdraw from family and society.\n\nAlzheimer\'s disease (AD) is a neurodegenerative disease and is the most common form of dementia, accounting for around 60–70% of cases. \nThe most common early symptom is difficulty in remembering recent events. \nAs the disease advances, symptoms can include problems with language, disorientation (including easily getting lost), mood swings, loss of motivation, self-neglect, and behavioral issues. \nAs a person\'s condition declines, they

In [24]:
query = 'Is there a cure for Alzheimer’s disease?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nAnswer the question based only on the following context:\nThere is no disease-modifying treatments proven to cure Alzheimer\'s disease and because of this, AD research has focused on interventions to prevent the onset and progression. There is no evidence that supports any particular measure in preventing AD, and studies of measures to prevent the onset or progression have produced inconsistent results. Epidemiological studies have proposed relationships between an individual\'s likelihood of developing AD and modifiable factors, such as medications,\n\nThere is no disease-modifying treatments proven to cure Alzheimer\'s disease and because of this, AD research has focused on interventions to prevent the onset and progression. There is no evidence that supports any particular measure in preventing AD, and studies of measures to prevent the onset or progression have produced inconsistent results. Epidemiological studies have proposed relationships between an individual\'s like

In [25]:
query = 'Can Alzheimer’s disease be prevented?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nAnswer the question based only on the following context:\nbut these have not yet stopped or reversed dementia. Hence, researchers increasingly believe that the best strategy is to prevent Alzheimer\'s by intervening before the brain has been irreversibly damaged.\n\nbut these have not yet stopped or reversed dementia. Hence, researchers increasingly believe that the best strategy is to prevent Alzheimer\'s by intervening before the brain has been irreversibly damaged.\n\nbut these have not yet stopped or reversed dementia. Hence, researchers increasingly believe that the best strategy is to prevent Alzheimer\'s by intervening before the brain has been irreversibly damaged.\nAnswer the question based on the above context: Can Alzheimer’s disease be prevented?.\nProvide a detailed answer.\nDon’t justify your answers.\nDon’t give information not mentioned in the CONTEXT INFORMATION.\nDo not say "according to the context" or "mentioned in the context" or similar.\n'

In [33]:
query = 'Does prevalence increase with age?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'System: You answer strictly using the given context.\nHuman: \nAnswer the question based only on the following context:\nprevalence was estimated to be 5.3% for those in the 60–74 age group, with the rate increasing to 13.8% in the 74–84 group and to 34.6% in those greater than 85. Prevalence rates in some less developed regions around the globe are lower. Both the prevalence and incidence rates of AD are steadily increasing, and the prevalence rate is estimated to triple by 2050 reaching 152 million, compared to the 50 million people with AD globally in 2020.\n\nprevalence was estimated to be 5.3% for those in the 60–74 age group, with the rate increasing to 13.8% in the 74–84 group and to 34.6% in those greater than 85. Prevalence rates in some less developed regions around the globe are lower. Both the prevalence and incidence rates of AD are steadily increasing, and the prevalence rate is estimated to triple by 2050 reaching 152 million, compared to the 50 million people with AD g

In [41]:
query = 'How does Alzheimer’s disease progress?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


"\nAnswer the question based only on the following context:\n\ngo through states of cognitive development, people with Alzheimer's disease go through the reverse process of progressive cognitive impairment.\n\ngo through states of cognitive development, people with Alzheimer's disease go through the reverse process of progressive cognitive impairment.\n\ngo through states of cognitive development, people with Alzheimer's disease go through the reverse process of progressive cognitive impairment.\n\nQuestion: How does Alzheimer’s disease progress?\n\nProvide a detailed answer.\nDon’t justify your answers.\nDon’t give information not mentioned in the context.\n"

In [51]:
query = 'Who first described Alzheimer’s disease?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nAnswer the question based only on the following context:\n[Document(id=\'b1e94aa3-b710-406f-b814-98867b2e41dd\', metadata={\'source\': \'text_files/alzheimers_2.txt\'}, page_content="Early onset\\nFurther information: Early-onset Alzheimer\'s disease"), Document(id=\'4facc8a3-cb1e-4215-b363-944748f32487\', metadata={\'source\': \'text_files/alzheimers_2.txt\'}, page_content="Early onset\\nFurther information: Early-onset Alzheimer\'s disease"), Document(id=\'8174b19c-e54a-4fb9-8a08-bec6f27accc5\', metadata={\'source\': \'text_files/alzheimers_2.txt\'}, page_content="Early onset\\nFurther information: Early-onset Alzheimer\'s disease")]\nAnswer the question based on the above context: Who first described Alzheimer’s disease?.\nProvide a detailed answer.\nDon’t justify your answers.\nDon’t give information not mentioned in the CONTEXT INFORMATION.\nDo not say "according to the context" or "mentioned in the context" or similar.\n'

In [52]:
query = 'What is early-onset Alzheimer’s disease?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nAnswer the question based only on the following context:\n[Document(id=\'b1e94aa3-b710-406f-b814-98867b2e41dd\', metadata={\'source\': \'text_files/alzheimers_2.txt\'}, page_content="Early onset\\nFurther information: Early-onset Alzheimer\'s disease"), Document(id=\'4facc8a3-cb1e-4215-b363-944748f32487\', metadata={\'source\': \'text_files/alzheimers_2.txt\'}, page_content="Early onset\\nFurther information: Early-onset Alzheimer\'s disease"), Document(id=\'8174b19c-e54a-4fb9-8a08-bec6f27accc5\', metadata={\'source\': \'text_files/alzheimers_2.txt\'}, page_content="Early onset\\nFurther information: Early-onset Alzheimer\'s disease")]\nAnswer the question based on the above context: What is early-onset Alzheimer’s disease?.\nProvide a detailed answer.\nDon’t justify your answers.\nDon’t give information not mentioned in the CONTEXT INFORMATION.\nDo not say "according to the context" or "mentioned in the context" or similar.\n'

In [53]:
query = 'What is a common cause of death in Alzheimer’s patients?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nAnswer the question based only on the following context:\n[Document(id=\'24bd9a86-67e1-41ff-bbbd-5a18962c203f\', metadata={\'source\': \'text_files/alzheimers_1.txt\'}, page_content="much more common symptoms. People with Alzheimer\'s disease will ultimately not be able to perform even the simplest tasks independently; muscle mass and mobility deteriorates to the point where they are bedridden and unable to feed themselves. The cause of death is usually an external factor, such as infection of pressure ulcers or pneumonia, not the disease itself. In some cases, there is a paradoxical lucidity immediately before death, where there is an unexpected recovery of mental clarity."), Document(id=\'3fe81d3d-dcf0-47e8-8221-bb1f6ecbaa40\', metadata={\'source\': \'text_files/alzheimers_1.txt\'}, page_content="much more common symptoms. People with Alzheimer\'s disease will ultimately not be able to perform even the simplest tasks independently; muscle mass and mobility deteriorates to t

In [54]:
query = 'Why is Alzheimer’s disease considered a growing global concern?'

rag_chain.invoke(query)

Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: \nAnswer the question based only on the following context:\n[Document(id=\'9557a7c9-232d-447c-ace8-c296311a9952\', metadata={\'source\': \'text_files/alzheimers_1.txt\'}, page_content="As of 2020, there were approximately 50 million people worldwide with Alzheimer\'s disease. It most often begins in people over 65 years of age, although up to 10% of cases are early-onset impacting those in their 30s to mid-60s. It affects about 6% of people 65 years and older, and women more often than men. The disease is named after German psychiatrist and pathologist Alois Alzheimer, who first described it in 1906. Alzheimer\'s financial burden on society is large, with an estimated global"), Document(id=\'ae7e9efb-e1df-40da-af01-a1856fc5b147\', metadata={\'source\': \'text_files/alzheimers_1.txt\'}, page_content="As of 2020, there were approximately 50 million people worldwide with Alzheimer\'s disease. It most often begins in people over 65 years of age, although up to 10% of cases are earl